# Telco Customer Churn Prediction with Apache Spark MLlib

This notebook loads the Telco churn CSV, explores the data, prepares features with Spark MLlib, trains two classifiers, and compares them for churn prediction.

In [18]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.functions import vector_to_array

try:
    spark = SparkSession.builder \
        .appName("TelcoChurnMLlib") \
        .config("spark.driver.memory", "2g") \
        .config("spark.sql.shuffle.partitions", "4") \
        .getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    print("Spark session started successfully")
except Exception as e:
    print(f"Error starting Spark: {e}")
    print("Ensure Java is installed and JAVA_HOME is set correctly.")
    raise

Spark session started successfully


In [19]:
import os

java_home = r"C:\Program Files\Java\jdk-21.0.11"
if os.path.exists(java_home):
    os.environ["JAVA_HOME"] = java_home
    os.environ["PATH"] = java_home + r"\bin;" + os.environ.get("PATH", "")
    print(f"Using JAVA_HOME: {java_home}")
else:
    print(f"WARNING: Java path not found: {java_home}")
    print("Please verify Java is installed at C:\\Program Files\\Java\\jdk-21.0.11")

Using JAVA_HOME: C:\Program Files\Java\jdk-21.0.11


## Load and Explore the Dataset

Load the Telco churn CSV and inspect its structure, data types, and data quality.

In [20]:
data_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
raw_df = spark.read.option("header", True).option("multiLine", False).csv(data_path)

raw_df.show(5, truncate=False)
raw_df.printSchema()
print("Rows:", raw_df.count())
print("Columns:", len(raw_df.columns))

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+-------------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines   |InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|Contract      |PaperlessBilling|PaymentMethod            |MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+-------------------------+--------------+------------+-----+
|7590-VHVEG|Female|0            |Yes    |No        |1     |No          |No phone service|DSL            |No            |Yes         |No              |N

In [21]:
columns_to_check = raw_df.columns
missing_counts = raw_df.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), c)).alias(c)
    for c in columns_to_check
])
missing_counts.show(truncate=False)

raw_df.describe().show()

+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|Contract|PaperlessBilling|PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|0         |0     |0            |0      |0         |0     |0           |0            |0              |0             |0           |0               |0          |0          |0              |0       |0               |0

## Data Preprocessing and Cleaning

Prepare the target column, clean `TotalCharges`, and create a few simple engineered features.

In [22]:
df = (
    raw_df
    .withColumnRenamed("customerID", "customer_id")
    .withColumn("label", F.when(F.col("Churn") == "Yes", F.lit(1.0)).otherwise(F.lit(0.0)))
    .withColumn("SeniorCitizen", F.col("SeniorCitizen").cast(IntegerType()))
    .withColumn("tenure", F.col("tenure").cast(DoubleType()))
    .withColumn("MonthlyCharges", F.col("MonthlyCharges").cast(DoubleType()))
    .withColumn(
        "TotalCharges",
        F.when(F.trim(F.col("TotalCharges")) == "", F.lit(0.0)).otherwise(F.trim(F.col("TotalCharges")).cast(DoubleType()))
    )
    .withColumn("charge_per_tenure", F.when(F.col("tenure") > 0, F.col("TotalCharges") / F.col("tenure")).otherwise(F.col("MonthlyCharges")))
    .withColumn("tenure_group", F.when(F.col("tenure") <= 12, "Short")
                .when(F.col("tenure") <= 48, "Medium")
                .otherwise("Long"))
    .drop("Churn")
)

df.select("customer_id", "label", "SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "charge_per_tenure", "tenure_group").show(5, truncate=False)

+-----------+-----+-------------+------+--------------+------------+-----------------+------------+
|customer_id|label|SeniorCitizen|tenure|MonthlyCharges|TotalCharges|charge_per_tenure|tenure_group|
+-----------+-----+-------------+------+--------------+------------+-----------------+------------+
|7590-VHVEG |0.0  |0            |1.0   |29.85         |29.85       |29.85            |Short       |
|5575-GNVDE |0.0  |0            |34.0  |56.95         |1889.5      |55.5735294117647 |Medium      |
|3668-QPYBK |1.0  |0            |2.0   |53.85         |108.15      |54.075           |Short       |
|7795-CFOCW |0.0  |0            |45.0  |42.3          |1840.75     |40.90555555555556|Medium      |
|9237-HQITU |1.0  |0            |2.0   |70.7          |151.65      |75.825           |Short       |
+-----------+-----+-------------+------+--------------+------------+-----------------+------------+
only showing top 5 rows


## Exploratory Data Analysis (EDA)

Summarize churn, service usage, and key numeric patterns.

In [23]:
df.groupBy("label").count().orderBy("label").show()

summary_cols = ["tenure", "MonthlyCharges", "TotalCharges", "charge_per_tenure"]
df.select([F.round(F.avg(c), 2).alias(c) for c in summary_cols]).show()

df.groupBy("Contract", "label").count().orderBy("Contract", "label").show()
df.groupBy("InternetService", "label").count().orderBy("InternetService", "label").show()

+-----+-----+
|label|count|
+-----+-----+
|  0.0| 5174|
|  1.0| 1869|
+-----+-----+

+------+--------------+------------+-----------------+
|tenure|MonthlyCharges|TotalCharges|charge_per_tenure|
+------+--------------+------------+-----------------+
| 32.37|         64.76|     2279.73|            64.76|
+------+--------------+------------+-----------------+

+--------------+-----+-----+
|      Contract|label|count|
+--------------+-----+-----+
|Month-to-month|  0.0| 2220|
|Month-to-month|  1.0| 1655|
|      One year|  0.0| 1307|
|      One year|  1.0|  166|
|      Two year|  0.0| 1647|
|      Two year|  1.0|   48|
+--------------+-----+-----+

+---------------+-----+-----+
|InternetService|label|count|
+---------------+-----+-----+
|            DSL|  0.0| 1962|
|            DSL|  1.0|  459|
|    Fiber optic|  0.0| 1799|
|    Fiber optic|  1.0| 1297|
|             No|  0.0| 1413|
|             No|  1.0|  113|
+---------------+-----+-----+



## Data Splitting and Scaling

Split the data and prepare Spark MLlib features for model training.

In [24]:
categorical_cols = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod", "tenure_group"
]
numeric_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "charge_per_tenure"]

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
ohe_cols = [f"{c}_ohe" for c in categorical_cols]
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=ohe_cols,
    dropLast=False,
    handleInvalid="keep"
)
assembler = VectorAssembler(inputCols=numeric_cols + ohe_cols, outputCol="features_raw", handleInvalid="keep")
preprocess = Pipeline(stages=indexers + [encoder, assembler])
preprocess_model = preprocess.fit(train_df)
train_prepared = preprocess_model.transform(train_df)
test_prepared = preprocess_model.transform(test_df)

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=False, withStd=True)
scaler_model = scaler.fit(train_prepared)
train_scaled = scaler_model.transform(train_prepared)
test_scaled = scaler_model.transform(test_prepared)

## Build Classification Models

Train Logistic Regression and Random Forest models with Spark MLlib.

In [25]:
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)
lr_model = lr.fit(train_scaled)

rf = RandomForestClassifier(featuresCol="features_raw", labelCol="label", seed=42, probabilityCol="probability")
rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [5, 10])
    .build()
)
rf_cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=rf_grid,
    evaluator=BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"),
    numFolds=3,
    seed=42
)
rf_cv_model = rf_cv.fit(train_prepared)

## Hyperparameter Tuning

A small cross-validation grid is used for the random forest model to improve robustness.

## Model Evaluation and Comparison

Compare the models using AUC-ROC, accuracy, and confusion-matrix style summaries.

In [26]:
auc_evaluator_lr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc_evaluator_rf = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="probability", metricName="areaUnderROC")
acc_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

lr_predictions = lr_model.transform(test_scaled)
rf_predictions = rf_cv_model.bestModel.transform(test_prepared)

results = [
    ("Logistic Regression", auc_evaluator_lr.evaluate(lr_predictions), acc_evaluator.evaluate(lr_predictions)),
    ("Random Forest", auc_evaluator_rf.evaluate(rf_predictions), acc_evaluator.evaluate(rf_predictions))
]

results_df = spark.createDataFrame(results, ["model", "auc_roc", "accuracy"])
results_df.show(truncate=False)

print("Logistic Regression confusion summary")
lr_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

print("Random Forest confusion summary")
rf_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

+-------------------+------------------+------------------+
|model              |auc_roc           |accuracy          |
+-------------------+------------------+------------------+
|Logistic Regression|0.8423894331156402|0.8066914498141264|
|Random Forest      |0.842486616464523 |0.7925650557620818|
+-------------------+------------------+------------------+

Logistic Regression confusion summary
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|  890|
|  0.0|       1.0|   86|
|  1.0|       0.0|  174|
|  1.0|       1.0|  195|
+-----+----------+-----+

Random Forest confusion summary
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|  886|
|  0.0|       1.0|   90|
|  1.0|       0.0|  189|
|  1.0|       1.0|  180|
+-----+----------+-----+



## Feature Importance Analysis

Interpret the tuned random forest model and identify the strongest churn drivers.

In [27]:
rf_best_model = rf_cv_model.bestModel
feature_attrs = train_prepared.schema["features_raw"].metadata["ml_attr"]["attrs"]
all_feature_attrs = [attr for group in feature_attrs.values() for attr in group]
feature_names = [attr.get("name", f"feature_{attr['idx']}") for attr in sorted(all_feature_attrs, key=lambda attr: attr["idx"])]

importance_data = sorted(
    [(name, float(score)) for name, score in zip(feature_names, rf_best_model.featureImportances.toArray())],
    key=lambda item: item[1],
    reverse=True
)

spark.createDataFrame(importance_data[:15], ["feature", "importance"]).show(truncate=False)

+----------------------------------+--------------------+
|feature                           |importance          |
+----------------------------------+--------------------+
|tenure                            |0.11725643700172372 |
|Contract_ohe_Month-to-month       |0.0991074028342623  |
|TotalCharges                      |0.09140059747880072 |
|MonthlyCharges                    |0.05537774670327661 |
|charge_per_tenure                 |0.05003565667731495 |
|InternetService_ohe_Fiber optic   |0.04959255480419497 |
|OnlineSecurity_ohe_No             |0.03984045839647444 |
|tenure_group_ohe_Short            |0.039483605319573574|
|TechSupport_ohe_No                |0.03765875397729899 |
|PaymentMethod_ohe_Electronic check|0.028888023145383794|
|OnlineBackup_ohe_No               |0.027864825592737452|
|tenure_group_ohe_Long             |0.02721928481808189 |
|Contract_ohe_Two year             |0.023556502508934426|
|InternetService_ohe_DSL           |0.019307829230078365|
|tenure_group_

## Predictions and Business Insights

Use the best model to rank the highest churn-risk customers and summarize what that means for retention.

In [ ]:
best_model_row = results_df.orderBy(F.desc("auc_roc")).first()
best_model_name = best_model_row["model"]
best_predictions = rf_predictions if best_model_name == "Random Forest" else lr_predictions

best_predictions.select(
    "customer_id",
    F.round(vector_to_array("probability")[1], 4).alias("churn_probability"),
    "prediction"
).orderBy(F.desc("churn_probability")).show(10, truncate=False)

print("Best model:", best_model_name)
print("Top retention focus: month-to-month customers with high monthly charges and shorter tenure.")

+-----------+-----------------+----------+
|customer_id|churn_probability|prediction|
+-----------+-----------------+----------+
|5150-ITWWB |0.856            |1.0       |
|4424-TKOPW |0.8381           |1.0       |
|8098-LLAZX |0.8363           |1.0       |
|1273-MTETI |0.8264           |1.0       |
|5192-EBGOV |0.8213           |1.0       |
|9124-LHCJQ |0.8194           |1.0       |
|0495-RVCBF |0.8184           |1.0       |
|6230-BSUXY |0.8164           |1.0       |
|3988-RQIXO |0.8163           |1.0       |
|2656-TABEH |0.8143           |1.0       |
+-----------+-----------------+----------+
only showing top 10 rows
Best model: Random Forest
Top retention focus: month-to-month customers with high monthly charges and shorter tenure.


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 49164)
Traceback (most recent call last):
  File "C:\Users\amare\AppData\Roaming\uv\python\cpython-3.11.14-windows-x86_64-none\Lib\socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "C:\Users\amare\AppData\Roaming\uv\python\cpython-3.11.14-windows-x86_64-none\Lib\socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "C:\Users\amare\AppData\Roaming\uv\python\cpython-3.11.14-windows-x86_64-none\Lib\socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "C:\Users\amare\AppData\Roaming\uv\python\cpython-3.11.14-windows-x86_64-none\Lib\socketserver.py", line 755, in __init__
    self.handle()
  File "d:\UK\apacheSpark\.venv\Lib\site-packages\pyspark\accumulators.py", line 303, in handle
    poll(accum_updates)
  File "d